# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TobyRathmell123/ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:

print("Rows:", len(df))
print("Base rate (declining rate):", df["is_declining"].mean().round(3))

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

RANDOM_STATE = 42
client_col = "client_hash_id"   # <-- this is the one that exists in your warehouse df

unique_clients = df[client_col].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)

test_client_count = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:test_client_count])

test_mask = df[client_col].isin(test_clients)
train_df = df[~test_mask].copy()
test_df = df[test_mask].copy()

print(f"Train: {len(train_df):,} rows, {train_df[client_col].nunique()} clients, declining rate {train_df['is_declining'].mean():.3f}")
print(f"Test:  {len(test_df):,} rows, {test_df[client_col].nunique()} clients, declining rate {test_df['is_declining'].mean():.3f}")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

FEATURES = ["impressions_prev30", "clicks_prev30", "avg_position_prev30",
            "active_days_prev30", "content_age_days"]

X_train = train_df[FEATURES].fillna(0)
y_train = train_df["is_declining"]
X_test = test_df[FEATURES].fillna(0)
y_test = test_df["is_declining"]

# ── Baseline: same idea as your w04 stale_visible_page rule ──────────
stale = (test_df["active_days_prev30"] <= 15).astype(int)
visible = (test_df["impressions_prev30"] >= 500).astype(int)
test_df["baseline_score"] = stale * visible * test_df["impressions_prev30"]

baseline_p50 = precision_at_k(test_df["is_declining"].values, test_df["baseline_score"].values, 50)

# ── Models ────────────────────────────────────────────────────────
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=50,
                                             class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                             class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    results[name] = precision_at_k(y_test.values, probs, 50)

# ── The comparison table ─────────────────────────────────────────
print(f"{'random (base rate)':22} Precision@50: {y_test.mean():.3f}")
print(f"{'baseline_rule':22} Precision@50: {baseline_p50:.3f}")
for name, p50 in results.items():
    print(f"{name:22} Precision@50: {p50:.3f}")

# ── Permutation importance on the best model ─────────────────────
best_name = max(results, key=results.get)
best_model = models[best_name]
print(f"\nBest model: {best_name}")

perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
importance_df = pd.DataFrame({"feature": FEATURES, "importance": perm.importances_mean}).sort_values("importance", ascending=False)
print(importance_df)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
test_df["rf_probability"] = models["random_forest"].predict_proba(X_test)[:, 1]

# False positives: model said "declining", actually wasn't
false_positives = test_df[(test_df["rf_probability"] >= 0.5) & (test_df["is_declining"] == 0)]
# False negatives: model missed a real decline
false_negatives = test_df[(test_df["rf_probability"] < 0.5) & (test_df["is_declining"] == 1)]

print(f"False positives: {len(false_positives)} | False negatives: {len(false_negatives)}")
false_positives[FEATURES + ["is_declining"]].head(5)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.